# Dask
This is a tutorial to use the Dask cluster from Jupyter, without Prefect.

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import *  

# Init cluster from this venv kernel
from resources.dask_clusters.dask_venv import init_dask_cluster_mockup_venv
dask_gateway, dask_cluster, dask_client = init_dask_cluster_mockup_venv(
    scale=1,
    worker_cores=1,
    worker_memory=2.0,
    scheduler_memory_limit=2,
)

In [ ]:
# Forward logging from dask workers to the caller.
# NOTE 1: we need to use the logging in the workers, "print" won't be forwarded.
# NOTE 2: we don't do this with prefect+dask because dask will write directly into the prefect logger.
dask_client.forward_logging()

In [ ]:
# Other imports
import logging
import os
import socket
from pathlib import Path

## 1. Implement the `Futures` tutorial: https://docs.dask.org/en/stable/futures.html
Note: this is how the `rs-server-staging` web service is using Dask.

In [ ]:
def get_ip_address() -> str:
    return socket.gethostbyname(socket.gethostname())

def inc(x, name):

    # From staging workers
    if name == "staging":
        # Just make sure that rs-server-staging is installed inside the dask workers.
        # NOTE: this import doesn't run the staging web service. 
        # It only imports its modules to be able to call the staging functions.
        # This is actually what the staging web service (that runs on another pod) is doing.
        from rs_server_staging import processors

    # From eopf workers, we can also import the eopf modules
    else:
        from eopf.product.eo_product import EOProduct        
    
    # Note that this is run from a dask worker with a different IP than the client,
    # and that the workers also differ between the staging and eopf workers.
    logging.warning(
        f"Hello from {os.environ['HELLO_FROM']!r} {get_ip_address()!r} ({name})")    

    return x + 1

def add(x, y):
    return x + y

# Set environment variable for the dask workers
def set_dask_env():
    os.environ["HELLO_FROM"] = "dask"

# Print client (=jupyter or terminal) IP address
logging.warning(f"Hello from 'client' {get_ip_address()!r}")

# Test this for the staging
client = dask_client
name = "mockup"
print(f"\nTest {name!r}:")

# Set environment variable for the dask workers
client.run(set_dask_env)

a = client.submit(inc, 10, name)  # calls inc(10) in background thread or process
b = client.submit(inc, 20, name)  # calls inc(20) in background thread or process
print(f"a: {a.result()}")
print(f"b: {b.result()}")

c = client.submit(add, a, b)  # calls add on the results of a and b
print(f"c: {c.result()}")

futures = client.map(inc, range(5), name=name)
results = client.gather(futures)  # this can be faster
print(results)

<div class="alert alert-info" role="alert">
Notes:

  1. Check in the logs that the client and dask clusters each run on **different** IP addresses.
      1. On kubernetes, you can run the `kubectl describe` command to check a pod IP address.
      1. In local mode, use: `docker inspect <container_id> | grep IPAddress`

## 2. `pip install` inside Dask workers

In [ ]:
# This method does not work in cluster mode, we don't know why. 
# The method in the next cell (using a .whl file) works.
if local_mode:

    # Test if a module is installed inside the dask workers
    def test_pip():
        import argh # yes this is a real module, see: https://pypi.org/project/argh/
        logging.warning(f" argh methods/attributes: {dir(argh)}")

    # The first time you will test this in workers, it will fail
    try:
        client.submit(test_pip).result()
    except ModuleNotFoundError:
        print("'argh' is not yet installed in the workers ...")

    # You can install it with: https://distributed.dask.org/en/stable/plugins.html#built-in-scheduler-plugins
    from dask.distributed import PipInstall
    plugin = PipInstall(packages=["argh"])
    client.register_plugin(plugin)

    # Now it will work.
    client.submit(test_pip, pure=False).result() # IMPORTANT: use pure=False to disable cache
    print("'argh' is now installed in the workers.")

In [ ]:
# Do the same with a wheel file. First download it.
whl_dir = "/tmp/emoji"
!rm -rf $whl_dir && mkdir -p $whl_dir && pip download --dest $whl_dir emoji
whl_file = os.listdir(whl_dir)[0]
whl_path = Path(whl_dir) / whl_file

# Then we'll upload and install it in the dask workers. 
# But it only works with .py, .egg or .zip
# See: https://distributed.dask.org/en/latest/api.html#distributed.Client.upload_file
# A .whl file is just a zip, so rename it.
whl_path = whl_path.rename(whl_path.with_suffix(".zip"))

In [ ]:
# Then do the same as before
def test_pip_whl():
    import emoji
    logging.warning(f" emoji methods/attributes: {dir(emoji)}")
try:
    client.submit(test_pip_whl, pure=False).result()
except ModuleNotFoundError:
    print("'emoji' is not yet installed in the workers ...")

# Upload the wheel/zip. It is automatically installed.
client.upload_file(str(whl_path))

client.submit(test_pip_whl, pure=False).result() # IMPORTANT: use pure=False to disable cache
print("'emoji' is now installed in the workers.")

## 3. Shutdown the dask clusters

In [ ]:
# Choose to shutdown the dask cluster
from resources.dask_clusters.dask_utils import *
shutdown_checkbox

In [ ]:
if shutdown_checkbox.value:
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)
    close_dask_clusters(dask_gateway, dask_cluster, client)